In [48]:
from dotenv import load_dotenv
from agents import Agent, Runner, function_tool
from pydantic import BaseModel, ValidationError, EmailStr, Field
import json
import gradio as gr

In [49]:
load_dotenv(override=True)

True

In [50]:
# Schema for the parser agent
class ParsedCustomerQuery(BaseModel):
    name: str
    email: str
    query: str


# Schema to validate the customer query
class CustomerQueryInput(BaseModel):
    name: str = Field(..., min_length=2, max_length=50)
    email: EmailStr
    query: str = Field(..., min_length=50, max_length=200)

In [51]:
# Using the pydantic model to validate the customer query

@function_tool
def validate_parsed_customer_query(name: str, email: str, query: str):
    try:
        validated = CustomerQueryInput(
            name=name.strip(),
            email=email.strip(),
            query=query.strip(),
        )
        print("Input has been successfully validated!")
        return validated.model_dump()
    except ValidationError as e:
        errors = []
        for error in e.errors():
            errors.append(f"{error['loc'][0]}: {error['msg']}")
        return errors

In [52]:
# Gradio interface for user interaction

async def process_customer_query(name: str, email: str, query: str):
    customer_query_payload = {
        "name": name.strip(),
        "email": email.strip(),
        "query": query.strip(),
    }

    validation_result = await Runner.run(
        query_validation_agent,
        input=json.dumps(customer_query_payload),
    )

    final_response_result = await Runner.run(
        final_response_agent,
        input=str(validation_result.final_output),
    )

    return final_response_result.final_output


css = """
@font-face {
    font-family: 'Waldenburg';
    src: local('Waldenburg');
    font-display: swap;
}

.gradio-container, .gradio-container * {
    font-family: 'Waldenburg', -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif !important;
}
"""

with gr.Blocks(title="Customer Query Intake") as demo:
    gr.Markdown("# Customer Query Form")
    gr.Markdown(
        "Please complete the form below so we can check whether your query is ready for processing."
    )

    name_input = gr.Textbox(
        label="Name",
        placeholder="Enter your full name",
    )
    email_input = gr.Textbox(
        label="Email",
        placeholder="Enter your email address",
    )
    query_input = gr.Textbox(
        label="Query",
        lines=8,
        placeholder="Describe the issue in detail.",
    )
    submit_button = gr.Button("Submit Query", variant="primary")
    final_output = gr.Textbox(label="Status", lines=5)

    submit_button.click(
        fn=process_customer_query,
        inputs=[name_input, email_input, query_input],
        outputs=final_output,
    )

demo.launch(css=css)

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


In [53]:
# Agent 1: parse messy customer text into a permissive dict-like object.
query_parsing_agent = Agent(
    "Query Parsing Agent",
    instructions=(
        "Extract the customer's name, email, and issue description from the input. "
        "Do not fix or invent missing values. If a value is missing, use an empty string."
    ),
    model="gpt-4o-mini",
    output_type=ParsedCustomerQuery,
)

In [54]:
# Agent 2: run deterministic validation through the validation tool.
query_validation_agent = Agent(
    "Query Validation Agent",
    instructions=(
        "You validate parsed customer query data by calling validate_parsed_customer_query. "
        "Return the validation result without adding extra assumptions."
    ),
    model="gpt-4o-mini",
    tools=[validate_parsed_customer_query],
)

In [55]:
# Agent 3: decide whether to accept the query or explain what needs fixing.
final_response_agent = Agent(
    "Customer Query Final Reviewer",
    instructions=(
        "You review customer query validation results and respond directly to the customer. "
        "If the validation result is accepted, say exactly: "
        "'Your query has been accepted and is under processing.' "
        "If validation fails, politely explain which details need to be corrected. "
        "Do not mention internal fields like success, true, false, validation, or greenlight."
    ),
    model="gpt-4o-mini",
)